# GraphRAG Pipeline (Indexing Time + Query Time)

This notebook documents and prototypes a GraphRAG workflow for global sensemaking over large corpora.

## Two-Phase Architecture
1. **Indexing Time**: convert raw text into a hierarchical graph index with community summaries.
2. **Query Time**: use map-reduce over community summaries to generate global answers.

## Phase 1: Indexing Time

### Step 1: Source Documents -> Text Chunks
Split the corpus into manageable chunks (for example, ~600 tokens).

### Step 2: Text Chunks -> Entities and Relationships
Use an LLM to extract entities, entity descriptions, relationships, and optional claims.

### Step 3: Entities and Relationships -> Knowledge Graph
Aggregate extracted triples into a single graph, reconcile duplicates, and weight edges by frequency.

### Step 4: Knowledge Graph -> Graph Communities
Partition the graph into modular communities (for example, Leiden community detection).

### Step 5: Graph Communities -> Community Summaries
Generate bottom-up summaries from leaf communities to higher-level communities recursively.

In [13]:
# Optional install (run once)
%pip install -U langchain langchain-community langchain-ollama networkx leidenalg igraph tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
!ollama ls

NAME                            ID              SIZE      MODIFIED     
openvoid/Void-Gemini:latest     ebade0d31690    -         15 hours ago    
gemini-3-flash-preview:cloud    ebade0d31690    -         15 hours ago    
qwen3-embedding:latest          64b933495768    4.7 GB    15 hours ago    
nomic-embed-text:latest         0a109f422b47    274 MB    4 weeks ago     
qwen3.5:397b-cloud              a7bf6f7891c3    -         5 weeks ago     
minimax-m2.5:cloud              c0d5751c800f    -         6 weeks ago     
glm-4.7:cloud                   023608864819    -         6 weeks ago     
glm-5:cloud                     c313cd065935    -         6 weeks ago     
kimi-k2-thinking:cloud          9752ffb77f53    -         8 weeks ago     
ministral-3:14b-cloud           615c59440878    -         8 weeks ago     
qwen3-vl:235b-instruct-cloud    2bf9522f6961    -         8 weeks ago     
kimi-k2.5:cloud                 6d1c3246c608    -         8 weeks ago     
deepseek-v3.1:671b-cloud    

In [15]:
from __future__ import annotations

import os
from typing import TypedDict, List
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END

# Local Ollama defaults
LOCAL_OLLAMA_BASE_URL = os.getenv("LOCAL_OLLAMA_BASE_URL", "http://localhost:11434")
LOCAL_LLM_MODEL = os.getenv("LOCAL_LLM_MODEL", "openvoid/Void-Gemini:latest")
llm = ChatOllama(model=LOCAL_LLM_MODEL, base_url=LOCAL_OLLAMA_BASE_URL, temperature=0)

print("Using local model:", LOCAL_LLM_MODEL)

Using local model: openvoid/Void-Gemini:latest


In [16]:
# =========================
# 1. STATE
# =========================

class GraphRAGState(TypedDict):
    query: str
    docs: List[Document]
    chunks: List[Document]

    graph_elements: List[str]
    communities: List[List[str]]
    summaries: List[str]

    partial_answers: List[str]
    final_answer: str


# =========================
# 3. LOAD + CHUNK PDF
# =========================
def load_and_chunk(pdf_path: str):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(docs)

    return docs, chunks


# =========================
# 4. EXTRACT GRAPH ELEMENTS
# =========================
def extract_graph(state: GraphRAGState):
    graph_elements = []

    for chunk in state["chunks"][:30]:
        result = llm.invoke(f"""
        Extract structured knowledge:

        - Entities
        - Relationships
        - Key facts

        Return concise bullet points.

        TEXT:
        {chunk.page_content}
        """).content

        graph_elements.append(result)

    return {"graph_elements": graph_elements}


# =========================
# 5. COMMUNITY DETECTION
# =========================
def detect_communities(state: GraphRAGState):
    elements = state["graph_elements"]

    # Simple grouping (replace with clustering later).
    communities = [elements[i:i + 5] for i in range(0, len(elements), 5)]

    return {"communities": communities}


# =========================
# 6. COMMUNITY SUMMARIZATION
# =========================
def summarize_communities(state: GraphRAGState):
    summaries = []

    for community in state["communities"]:
        summary = llm.invoke(f"""
        Summarize this knowledge cluster clearly:

        {community}

        Focus on:
        - main ideas
        - connections
        - themes
        """).content

        summaries.append(summary)

    return {"summaries": summaries}

## Phase 2: Query Time (Map-Reduce)

### Map Step
At a selected hierarchy level, split community summaries into batches and produce partial answers in parallel.
Each partial answer receives a helpfulness score.

### Reduce Step
Filter/sort partial answers by score, pack them into a final context window until token budget is reached, then synthesize one global answer.

In [17]:
# =========================
# 7. MAP STEP (Partial Answers)
# =========================

def map_answers(state: GraphRAGState):
    partial_answers = []

    for summary in state["summaries"]:
        ans = llm.invoke(f"""
        Use this summary to answer the query.

        QUERY:
        {state['query']}

        SUMMARY:
        {summary}

        Also include a relevance score (0-100).
        """).content

        partial_answers.append(ans)

    return {"partial_answers": partial_answers}


# =========================
# 8. REDUCE STEP (Final Answer)
# =========================
def reduce_answer(state: GraphRAGState):
    combined = "\n\n".join(state["partial_answers"])

    final = llm.invoke(f"""
    Combine the following into one final answer.

    Remove redundancy.
    Keep important insights.

    {combined}
    """).content

    return {"final_answer": final}


# =========================
# 9. BUILD GRAPH WORKFLOW
# =========================
def build_graph_rag():
    builder = StateGraph(GraphRAGState)

    builder.add_node("extract_graph", extract_graph)
    builder.add_node("detect_communities", detect_communities)
    builder.add_node("summarize_communities", summarize_communities)
    builder.add_node("map_answers", map_answers)
    builder.add_node("reduce_answer", reduce_answer)

    builder.add_edge(START, "extract_graph")
    builder.add_edge("extract_graph", "detect_communities")
    builder.add_edge("detect_communities", "summarize_communities")
    builder.add_edge("summarize_communities", "map_answers")
    builder.add_edge("map_answers", "reduce_answer")
    builder.add_edge("reduce_answer", END)

    return builder.compile()


# =========================
# 10. MAIN APP (Notebook Run)
# =========================
PDF_PATH = os.getenv("GRAPHRAG_PDF_PATH", r"F:\resume.pdf")

print("Loading PDF...")
docs, chunks = load_and_chunk(PDF_PATH)

print("Building GraphRAG workflow...")
app = build_graph_rag()

print("GraphRAG Ready! Ask questions (type 'exit' to quit)")

while True:
    query = input("Ask: ").strip()

    if query.lower() == "exit":
        break

    if not query:
        print("Please enter a query.")
        continue

    result = app.invoke({
        "query": query,
        "docs": docs,
        "chunks": chunks
    })

    print("\nFINAL ANSWER:\n")
    print(result["final_answer"])
    print("\n" + "=" * 60 + "\n")

Loading PDF...
Building GraphRAG workflow...
GraphRAG Ready! Ask questions (type 'exit' to quit)

FINAL ANSWER:

Hello, **Sajid Miya**! You are a high-achieving Computer Science and Engineering student at the Thapar Institute of Engineering and Technology and an aspiring Machine Learning engineer. With a strong technical foundation in Python, full-stack development, and AI, you have demonstrated your expertise through impactful projects like your real-time American Sign Language (ASL) translator.

**Relevance Score:** 100/100



FINAL ANSWER:

Sajid Miya is a highly qualified candidate for AI engineering (**Relevance Score: 95/100**), combining a strong academic foundation (BE in Computer Science, 8.9 CGPA) with specialized technical expertise. He is proficient in **Python, C++, and C**, with a primary focus on **Machine Learning and Computer Vision** using tools like **OpenCV and MediaPipe**. His practical experience is highlighted by his development of an **AI-powered ASL Translator*

## Why GraphRAG for Global Questions?

Conventional vector RAG typically retrieves only a small set of local chunks.
GraphRAG can reason over the broader dataset through hierarchical community summaries, improving answer breadth and thematic coverage for global sensemaking tasks.

## Next Implementation Upgrades
1. Add tokenizer-aware chunking (token, not character based).
2. Add structured extraction schema (entities, relations, claims).
3. Add real graph construction with weighted edges in `networkx` or a graph DB.
4. Add Leiden community detection and hierarchical clustering metadata.
5. Add local Ollama prompts for map and reduce generation with score calibration.